# Marco 1 — Importação local e SQLite

Secretária IA, projeto acadêmico. Execute todas as células com o ambiente Python 3.12 preparado conforme README. Não usa OpenAI, LangSmith nem qualquer API. O núcleo é parsing determinístico e SQLite, ainda não uma chain/agente.

Somente dados fictícios são lidos. O banco desta demonstração é temporário e independente; executar novamente reconstrói o exemplo sem apagar um banco do usuário. O terminal usa um arquivo persistente em dados/privados.


In [1]:
from pathlib import Path
import json, sys, tempfile

# CAMINHOS PORTÁTEIS: localiza o projeto no diretório atual ou em seus pais, sem caminho pessoal.
raiz = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidato in (base, base / "Agentes Cognitivos com LangChain" / "Projeto final 2"):
        if (candidato / "PLANO_PROJETO_SECRETARIA_IA.md").is_file() and (candidato / "src/secretaria").is_dir():
            raiz = candidato
            break
    if raiz:
        break
if raiz is None:
    raise RuntimeError("Abra o notebook com a pasta completa do projeto e seu ambiente Python.")
sys.path.insert(0, str(raiz / "src"))
from secretaria.importador import ler_arquivo, interpretar
from secretaria.banco import Banco, Contexto
print("Python:", sys.version.split()[0])


Python: 3.12.6


## 1. Ler mensagens e preservar contexto

O cabeçalho inicia uma mensagem; linhas seguintes continuam o texto. Avisos, mídia omitida e mensagens removidas são identificados sem inventar conteúdo.


In [2]:
# PARSING: o gabarito é independente da interpretação, criado junto do corpus fictício.
arquivo = raiz / "dados/ficticios/conversa_empresa.txt"
gabarito = json.loads((raiz / "dados/ficticios/gabarito_importacao.json").read_text(encoding="utf-8"))
fonte = ler_arquivo(arquivo)
leitura = interpretar(fonte.texto)
assert len(leitura.mensagens) == gabarito["cabecalhos"]
assert leitura.mensagens[2].texto == gabarito["mensagem_multilinha"]["texto"]
print("Mensagens lidas:", len(leitura.mensagens), "| Rejeições:", len(leitura.rejeicoes))
print(leitura.mensagens[2].texto)


Mensagens lidas: 40 | Rejeições: 0
Projeto Alfa: implantação do portal de atendimento.
Cliente: Alfa Serviços.
Responsável: Ana.


## 2. Criar banco e importar com autoria

A seleção de Ana é identidade de demonstração, não login real. O corpus bruto é privado do importador, com acesso do administrador. A autorização completa sobre dados operacionais pertence às próximas etapas.


In [3]:
# SQLITE/MIGRAÇÃO: o arquivo e o esquema nascem automaticamente, com FKs e transações.
ambiente_demo = tempfile.TemporaryDirectory(prefix="secretaria_notebook_")
caminho_banco = Path(ambiente_demo.name) / "demo.sqlite3"
ctx = Contexto("demo", "ana")
with Banco(caminho_banco) as banco:
    banco.preparar_demo()
    corpus_id = banco.selecionar_corpus(ctx, "Empresa fictícia")
    primeira = banco.importar(ctx, corpus_id, arquivo)
    assert primeira["novas"] == 40
    print(primeira)


{'importacao_id': 1, 'corpus_id': 1, 'reimportacao': False, 'lidas': 40, 'novas': 40, 'reutilizadas': 0, 'rejeicoes': 0, 'avisos': 1, 'tipos': {'sistema': 1, 'mensagem': 37, 'midia_omitida': 1, 'removida': 1}, 'formatos': ['ios'], 'fuso': 'America/Sao_Paulo', 'origem_fuso': 'premissa configurada; não informada no arquivo', 'primeiro_instante_utc': '2026-11-09T11:00:00+00:00', 'ultimo_instante_utc': '2026-11-09T11:39:00+00:00'}


## 3. Reimportar sem duplicar

O hash do conteúdo impede repetir o lote. A identidade das mensagens combina corpus, data, autor, conteúdo e ocorrência, preservando duas mensagens legítimas idênticas.


In [4]:
# PERSISTÊNCIA: reabre o mesmo arquivo após fechar a conexão anterior.
with Banco(caminho_banco) as banco:
    segunda = banco.importar(ctx, corpus_id, arquivo)
    assert segunda["reimportacao"] and segunda["novas"] == 0
    assert banco.contar(ctx, corpus_id) == 40
    repetidas = [m for m in banco.listar(ctx, corpus_id) if m["texto"] == gabarito["repeticao_legitima"]["texto"]]
    assert len(repetidas) == 2
    print("Novas na segunda importação:", segunda["novas"])
    print("Mensagens preservadas:", banco.contar(ctx, corpus_id))
    print("Ocorrências legítimas idênticas:", len(repetidas))


Novas na segunda importação: 0
Mensagens preservadas: 40
Ocorrências legítimas idênticas: 2


## 4. Mesmo TXT em ZIP e rastreabilidade

O ZIP é lido em memória e anexos não são extraídos. Os avisos de edição não alteram o original nem escolhem automaticamente qual interpretação vale.


In [5]:
from zipfile import ZipFile
# FONTES: ZIP e TXT com o mesmo conteúdo apontam ao mesmo lote do corpus.
zip_demo = Path(ambiente_demo.name) / "exportacao.zip"
with ZipFile(zip_demo, "w") as z:
    z.write(arquivo, arcname="_chat.txt")
with Banco(caminho_banco) as banco:
    terceira = banco.importar(ctx, corpus_id, zip_demo)
    assert terceira["reimportacao"] and banco.contar(ctx, corpus_id) == 40
    avisos = banco.avisos(ctx, corpus_id, primeira["importacao_id"])
    assert any(a["tipo"] == "possivel_edicao" for a in avisos)
    print("ZIP repetido:", terceira["novas"], "novas mensagens")
    print("Avisos:", avisos)


ZIP repetido: 0 novas mensagens
Avisos: [{'tipo': 'possivel_edicao', 'detalhes': {'mensagem_nova': 30, 'candidatas_anteriores': [29], 'motivo': 'Edição sinalizada; original preservado e resolução manual pendente.'}}]


## O que foi demonstrado e o que falta

Demonstrado: parsing, fonte/linhas, criação do SQLite, autoria de importação, persistência entre conexões, repetição sem duplicar e leitura segura de ZIP. Não foi demonstrada memória cognitiva, compreensão por LLM, agenda, financeiro operacional, RAG, API externa nem LangSmith. A persistência aqui não comprova a conservação de um runtime temporário após reiniciar o notebook.

Próximo incremento: propostas estruturadas a partir do corpus fictício, sem gravação automática de fatos operacionais.
